# Dataset processing for classification model

 it uses **paths relative to the GitHub repository root** (no `C:\Users\...` paths).
Expected repository structure (recommended):

- `data/` : input files (COCO JSON, images, raw labels if any)
- `outputs/` : generated files (reindexed COCO, YOLO labels, train/val split)
- `notebooks/` : this notebook
- `src/` : reusable scripts

> If your images/labels are too large or confidential, keep them **outside git** and point `DATA_DIR` to a local folder (still without absolute hard-coding in many cells).


In [ ]:
# Imports (standard)
from __future__ import annotations

import os
import json
import shutil
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

# Optional (visual checks)
import matplotlib.pyplot as plt

# Optional (progress bar)
try:
    from tqdm import tqdm
except ImportError:
    tqdm = lambda x, **k: x  # fallback

# For notebook display
try:
    from IPython.display import display
except ImportError:
    def display(x):
        print(x)


In [ ]:
# ------------------------------------------------------------
# 0) REPO-ROOT + PATH CONFIG 
# ------------------------------------------------------------

def find_repo_root(start: Path | None = None) -> Path:
    """Find the repository root by searching for a `.git` folder or a `README.md`.
    Works whether you run the notebook from repo root or from `notebooks/`.
    """
    start = (start or Path.cwd()).resolve()
    for p in [start] + list(start.parents):
        if (p / ".git").exists() or (p / "README.md").exists():
            return p
    # fallback: current directory
    return start

REPO_ROOT = find_repo_root()
DATA_DIR  = REPO_ROOT / "data"
OUT_DIR   = REPO_ROOT / "outputs"

# Input files (edit names if needed)
COCO_INPUT_JSON   = DATA_DIR / "final_data.json"            # raw export from Hasty (COCO)
COCO_REINDEX_JSON = OUT_DIR / "dataset_coco_reindexed.json" # generated (reindexed categories)
CLASSES_TXT       = OUT_DIR / "classes.txt"                 # generated from COCO categories
ANNOTATIONS_CSV   = OUT_DIR / "annotations.csv"             # generated from COCO annotations

# Images directory (the path referenced by COCO "file_name")
# Option A: images are stored in `data/Phototheque/...`
IMAGES_DIR = DATA_DIR / "Phototheque"

# YOLO labels output folder (one .txt per image)
YOLO_LABELS_DIR = OUT_DIR / "yolo_labels"

# Train/val output structure for detection/segmentation training
YOLO_DATASET_DIR = OUT_DIR / "yolo_dataset"
IMG_TRAIN_DIR = YOLO_DATASET_DIR / "images" / "train"
IMG_VAL_DIR   = YOLO_DATASET_DIR / "images" / "val"
LAB_TRAIN_DIR = YOLO_DATASET_DIR / "labels" / "train"
LAB_VAL_DIR   = YOLO_DATASET_DIR / "labels" / "val"

# Parameters
TRAIN_RATIO = 0.8
VAL_RATIO   = 0.2
MAX_VAL_RATIO_PER_CLASS = 0.20  # post-processing constraint (best-effort)
RARE_THRESHOLD = 10
RARE_CLASS_NAME = "rare_species"

# Create output dirs (safe)
for p in [OUT_DIR, YOLO_LABELS_DIR, IMG_TRAIN_DIR, IMG_VAL_DIR, LAB_TRAIN_DIR, LAB_VAL_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT)
print("DATA_DIR :", DATA_DIR)
print("OUT_DIR  :", OUT_DIR)


## 1. COCO utilities (load, reindex categories)

Hasty exports sometimes have category IDs not starting at 0.
YOLO expects class IDs starting at 0 (contiguous).


In [ ]:
def load_coco_json(path: Path) -> dict:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"COCO JSON not found: {path}")
    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)
    required = {"images", "annotations", "categories"}
    missing = required - set(data.keys())
    if missing:
        raise ValueError(f"Missing keys in COCO JSON: {missing}")
    return data

def reindex_coco_categories(input_json: Path, output_json: Path) -> dict:
    """Reindex COCO category IDs to [0..K-1] and update annotation category_id."""
    coco = load_coco_json(input_json)

    cats = sorted(coco["categories"], key=lambda x: x["id"])
    old_to_new = {c["id"]: i for i, c in enumerate(cats)}

    # Update categories
    new_categories = []
    for c in cats:
        new_c = dict(c)
        new_c["id"] = old_to_new[c["id"]]
        new_categories.append(new_c)

    # Update annotations
    for ann in coco["annotations"]:
        ann["category_id"] = old_to_new[ann["category_id"]]

    coco["categories"] = new_categories

    output_json = Path(output_json)
    output_json.parent.mkdir(parents=True, exist_ok=True)
    with output_json.open("w", encoding="utf-8") as f:
        json.dump(coco, f, ensure_ascii=False, indent=2)

    print(f" Reindexed COCO saved to: {output_json}")
    print(f"   #categories: {len(coco['categories'])} | #images: {len(coco['images'])} | #annotations: {len(coco['annotations'])}")
    return coco


In [ ]:
# Run reindexing (skip gracefully if input not present)
if COCO_INPUT_JSON.exists():
    coco_data = reindex_coco_categories(COCO_INPUT_JSON, COCO_REINDEX_JSON)
else:
    coco_data = None
    print(f" COCO_INPUT_JSON not found: {COCO_INPUT_JSON}\n   -> Put your file there or edit COCO_INPUT_JSON in the config cell.")


## 2. Export `classes.txt` from COCO categories

In [ ]:
def export_classes_txt(coco: dict, output_txt: Path) -> dict[int, str]:
    """Write classes.txt with one line per class (id: 'name'). Returns mapping id->name."""
    output_txt = Path(output_txt)
    output_txt.parent.mkdir(parents=True, exist_ok=True)

    id_to_name = {}
    cats = sorted(coco["categories"], key=lambda x: x["id"])
    with output_txt.open("w", encoding="utf-8") as f:
        for c in cats:
            cid = int(c["id"])
            name = str(c.get("name", f"class_{cid}"))
            id_to_name[cid] = name
            f.write(f"{cid}: '{name}'\n")
    print(f" classes.txt written: {output_txt}")
    return id_to_name

if coco_data is not None:
    id_to_name = export_classes_txt(coco_data, CLASSES_TXT)
else:
    id_to_name = None


## 3. Convert COCO → YOLO segmentation labels + `annotations.csv`

Output YOLO label format (segmentation):
`class_id x1 y1 x2 y2 ...` with coordinates **normalized** to [0,1].

We also generate a CSV with (image_id, file_name, class_id) to support splitting.


In [ ]:
def coco_to_yolo_segmentation(
    coco: dict,
    yolo_labels_dir: Path,
    annotations_csv: Path
) -> pd.DataFrame:
    yolo_labels_dir = Path(yolo_labels_dir)
    yolo_labels_dir.mkdir(parents=True, exist_ok=True)

    # image_id -> (file_name, width, height)
    images = {img["id"]: img for img in coco["images"]}

    # Prepare dataframe rows
    rows = []

    # group annotations by image_id
    anns_by_img = defaultdict(list)
    for ann in coco["annotations"]:
        anns_by_img[ann["image_id"]].append(ann)

    for image_id, anns in tqdm(anns_by_img.items(), desc="COCO→YOLO"):
        img = images.get(image_id)
        if img is None:
            continue

        file_name = img["file_name"]
        w, h = img.get("width"), img.get("height")
        if not w or not h:
            # skip if missing size
            continue

        # YOLO label file name must match image base name
        stem = Path(file_name).stem
        label_path = yolo_labels_dir / f"{stem}.txt"

        yolo_lines = []
        for ann in anns:
            cid = int(ann["category_id"])
            seg = ann.get("segmentation", [])
            if not seg:
                continue

            # COCO polygon segmentation can be list of lists
            # We'll keep each polygon as a separate line
            for poly in seg:
                if not poly or len(poly) < 6:
                    continue
                # poly: [x1,y1,x2,y2,...] in absolute pixels
                coords = np.array(poly, dtype=float).reshape(-1, 2)
                coords[:, 0] = coords[:, 0] / w
                coords[:, 1] = coords[:, 1] / h

                # clamp to [0,1] for safety
                coords = np.clip(coords, 0.0, 1.0)

                flat = coords.reshape(-1)
                flat_str = " ".join([f"{v:.6f}" for v in flat])
                yolo_lines.append(f"{cid} {flat_str}")

                rows.append({
                    "image_id": image_id,
                    "file_name": file_name,
                    "class_id": cid
                })

        if yolo_lines:
            label_path.write_text("\n".join(yolo_lines) + "\n", encoding="utf-8")

    df = pd.DataFrame(rows).drop_duplicates()
    annotations_csv = Path(annotations_csv)
    annotations_csv.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(annotations_csv, index=False)
    print(f" YOLO labels written to: {yolo_labels_dir}")
    print(f" annotations.csv written: {annotations_csv} | rows={len(df)}")
    return df

if coco_data is not None:
    ann_df = coco_to_yolo_segmentation(coco_data, YOLO_LABELS_DIR, ANNOTATIONS_CSV)
else:
    ann_df = None


## 4. Quick checks

- Show a few classes and counts
- Confirm labels exist


In [ ]:
if ann_df is not None and not ann_df.empty:
    print("Unique images:", ann_df["file_name"].nunique())
    print("Unique classes:", ann_df["class_id"].nunique())
    display(ann_df.head())
    class_counts = ann_df["class_id"].value_counts().sort_index()
    display(class_counts.head(10))
else:
    print("No annotations dataframe available yet.")


## 5. Multilabel stratified split (images + labels)

This split is for **detection/segmentation** datasets where one image can contain multiple classes.
We stratify on the presence/absence matrix (image × class).

Requires: `iterative-stratification`
Install (once): `pip install iterative-stratification`


In [ ]:
# Try importing the splitter (gracefully)
try:
    from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
except ImportError:
    MultilabelStratifiedShuffleSplit = None
    print(" Missing dependency: iterative-stratification")
    print("   Install with: pip install iterative-stratification")


In [ ]:
def build_image_multilabel_matrix(df: pd.DataFrame):
    """From annotations.csv rows, build:
    - all_images: list of file_name
    - class_ids: sorted list of class ids
    - Y: binary matrix (n_images, n_classes)
    - image_to_classes: mapping file_name -> set(class_ids)
    """
    image_to_classes = defaultdict(set)
    for _, r in df.iterrows():
        image_to_classes[r["file_name"]].add(int(r["class_id"]))

    all_images = sorted(image_to_classes.keys())
    class_ids = sorted({c for s in image_to_classes.values() for c in s})
    class_to_col = {c: i for i, c in enumerate(class_ids)}

    Y = np.zeros((len(all_images), len(class_ids)), dtype=int)
    for i, fn in enumerate(all_images):
        for c in image_to_classes[fn]:
            Y[i, class_to_col[c]] = 1

    return all_images, class_ids, Y, image_to_classes, class_to_col

def limit_val_ratio_per_class(train_files, val_files, image_to_classes, class_to_col, max_ratio=0.20):
    """Best-effort post-processing: move images from val→train if a class exceeds max_ratio in val."""
    nb_classes = len(class_to_col)

    changed = True
    while changed:
        changed = False

        train_counts = np.zeros(nb_classes, dtype=int)
        val_counts   = np.zeros(nb_classes, dtype=int)

        for f in train_files:
            for c in image_to_classes[f]:
                train_counts[class_to_col[c]] += 1
        for f in val_files:
            for c in image_to_classes[f]:
                val_counts[class_to_col[c]] += 1

        for c, idx in class_to_col.items():
            total = train_counts[idx] + val_counts[idx]
            if total == 0:
                continue
            ratio = val_counts[idx] / total
            if ratio > max_ratio:
                # move one image that contains class c
                candidate = next((f for f in val_files if c in image_to_classes[f]), None)
                if candidate is None:
                    continue
                val_files.remove(candidate)
                train_files.append(candidate)
                changed = True
                break

    return train_files, val_files


In [ ]:
def find_image_path(images_root: Path, file_name: str) -> Path | None:
    """Search for file_name under images_root (recursive)."""
    images_root = Path(images_root)
    for p in images_root.rglob(file_name):
        if p.is_file():
            return p
    return None

def copy_image_and_label(file_name: str, dst_img_dir: Path, dst_lab_dir: Path):
    src_img = find_image_path(IMAGES_DIR, Path(file_name).name)
    if src_img is None:
        print(f" Image not found: {file_name}")
    else:
        shutil.copy(src_img, Path(dst_img_dir) / src_img.name)

    stem = Path(file_name).stem
    src_lab = YOLO_LABELS_DIR / f"{stem}.txt"
    if src_lab.exists():
        shutil.copy(src_lab, Path(dst_lab_dir) / src_lab.name)
    else:
        print(f" Label not found: {src_lab.name}")

def run_multilabel_split(df: pd.DataFrame):
    if MultilabelStratifiedShuffleSplit is None:
        raise RuntimeError("iterative-stratification not installed.")
    all_images, class_ids, Y, image_to_classes, class_to_col = build_image_multilabel_matrix(df)

    msss = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=VAL_RATIO, random_state=42)
    X = np.arange(len(all_images)).reshape(-1, 1)

    for train_idx, val_idx in msss.split(X, Y):
        train_files = [all_images[i] for i in train_idx]
        val_files   = [all_images[i] for i in val_idx]

    print(f"Initial split: train={len(train_files)}, val={len(val_files)}")

    train_files, val_files = limit_val_ratio_per_class(
        train_files, val_files, image_to_classes, class_to_col, max_ratio=MAX_VAL_RATIO_PER_CLASS
    )
    print(f"After post-processing: train={len(train_files)}, val={len(val_files)}")

    # copy
    for fn in train_files:
        copy_image_and_label(fn, IMG_TRAIN_DIR, LAB_TRAIN_DIR)
    for fn in val_files:
        copy_image_and_label(fn, IMG_VAL_DIR, LAB_VAL_DIR)

    return train_files, val_files, class_ids

if ann_df is not None and not ann_df.empty and MultilabelStratifiedShuffleSplit is not None:
    train_files, val_files, class_ids = run_multilabel_split(ann_df)
    print(" Train/val copy finished.")
else:
    print("Skip split: missing ann_df or iterative-stratification.")


## 6. Orphan check (labels without images / images without labels)

This should run without error even if folders are empty.


In [ ]:
def orphan_check(labels_dir: Path, images_dir: Path, out_txt: Path):
    labels_dir = Path(labels_dir)
    images_dir = Path(images_dir)

    label_stems = {p.stem for p in labels_dir.glob("*.txt")}
    image_stems = {p.stem for p in images_dir.glob("*") if p.suffix.lower() in {".jpg",".jpeg",".png",".bmp",".tiff"}}

    labels_without_images = sorted(label_stems - image_stems)
    images_without_labels = sorted(image_stems - label_stems)

    if labels_without_images:
        print("Labels without images:", len(labels_without_images))
    if images_without_labels:
        print("Images without labels:", len(images_without_labels))

    out_txt = Path(out_txt)
    out_txt.parent.mkdir(parents=True, exist_ok=True)
    with out_txt.open("w", encoding="utf-8") as f:
        f.write("labels_without_images\n")
        for s in labels_without_images:
            f.write(s + "\n")
        f.write("\nimages_without_labels\n")
        for s in images_without_labels:
            f.write(s + "\n")

    print(f"✅ Orphan report written: {out_txt}")
    return labels_without_images, images_without_labels

_ = orphan_check(LAB_TRAIN_DIR, IMG_TRAIN_DIR, OUT_DIR / "orphans_train.txt")
_ = orphan_check(LAB_VAL_DIR,   IMG_VAL_DIR,   OUT_DIR / "orphans_val.txt")


## 7. Optional step : merge rare classes into a single class (`rare_species`)

This rewrites:
- a new `classes_final.txt`
- updated YOLO label files (in-place) in a chosen labels directory.

Use carefully; ideally run on a **copy** of the labels.


In [ ]:
def parse_classes_file(classes_path: Path) -> dict[int, str]:
    id_to_species = {}
    with Path(classes_path).open("r", encoding="utf-8") as f:
        for line in f:
            if ":" in line:
                k, v = line.strip().split(":", 1)
                cid = int(k.strip())
                name = v.strip().strip("'").strip('"')
                id_to_species[cid] = name
    return id_to_species

def merge_rare_classes(labels_dir: Path, classes_path: Path, output_classes_path: Path,
                       rare_threshold: int = 10, rare_name: str = "rare_species"):
    labels_dir = Path(labels_dir)
    id_to_species = parse_classes_file(classes_path)

    # Count annotations per class
    counts = Counter()
    for p in labels_dir.glob("*.txt"):
        for line in p.read_text(encoding="utf-8").splitlines():
            parts = line.strip().split()
            if parts:
                counts[int(parts[0])] += 1

    rare_ids = {cid for cid, c in counts.items() if c < rare_threshold}
    print(f"Rare class IDs (<{rare_threshold}): {sorted(rare_ids)}")

    # Build new mapping
    old_to_new = {}
    new_id_to_species = {}
    new_id = 0

    for old_id in sorted(id_to_species.keys()):
        if old_id not in rare_ids and counts.get(old_id, 0) >= rare_threshold:
            old_to_new[old_id] = new_id
            new_id_to_species[new_id] = id_to_species[old_id]
            new_id += 1

    rare_new_id = new_id
    new_id_to_species[rare_new_id] = rare_name
    for old_id in id_to_species.keys():
        if old_id in rare_ids or counts.get(old_id, 0) < rare_threshold:
            old_to_new[old_id] = rare_new_id

    # Write new classes file
    output_classes_path = Path(output_classes_path)
    with output_classes_path.open("w", encoding="utf-8") as f:
        for i in range(len(new_id_to_species)):
            f.write(f"{i}: '{new_id_to_species[i]}'\n")

    # Rewrite labels
    for p in labels_dir.glob("*.txt"):
        new_lines = []
        for line in p.read_text(encoding="utf-8").splitlines():
            parts = line.strip().split()
            if not parts:
                continue
            old_id = int(parts[0])
            parts[0] = str(old_to_new.get(old_id, old_id))
            new_lines.append(" ".join(parts))
        p.write_text("\n".join(new_lines) + "\n", encoding="utf-8")

    print(f" Updated labels in: {labels_dir}")
    print(f" New classes file: {output_classes_path}")
    return old_to_new, new_id_to_species

# Example (commented):
# old_to_new, new_id_to_species = merge_rare_classes(
#     labels_dir=YOLO_LABELS_DIR,
#     classes_path=CLASSES_TXT,
#     output_classes_path=OUT_DIR / "classes_final.txt",
#     rare_threshold=RARE_THRESHOLD,
#     rare_name=RARE_CLASS_NAME
# )


## 8. list images that contain selected class IDs

Useful for quick checks against Hasty.


In [ ]:
def images_containing_classes(labels_dir: Path, images_dir: Path, target_classes: set[str], image_exts=(".jpg",".jpeg",".png")):
    labels_dir = Path(labels_dir)
    images_dir = Path(images_dir)

    matches = []
    for p in labels_dir.glob("*.txt"):
        txt = p.read_text(encoding="utf-8")
        found = False
        for line in txt.splitlines():
            parts = line.strip().split()
            if parts and parts[0] in target_classes:
                found = True
                break
        if found:
            # try to find image with same stem
            stem = p.stem
            img = None
            for ext in image_exts:
                candidate = find_image_path(images_dir, stem + ext)
                if candidate:
                    img = candidate.name
                    break
            matches.append(img or stem)

    print(f"{len(matches)} images found containing classes: {sorted(target_classes)}")
    for m in matches[:50]:
        print(m)
    if len(matches) > 50:
        print("... (truncated)")
    return matches

# Example:
# _ = images_containing_classes(YOLO_LABELS_DIR, IMAGES_DIR, target_classes={'112','80'})
